In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import tensorflow as tf
#import tensorflow_addons as tfa
import matplotlib.pyplot as plt
import numpy as np
from transformers import BertTokenizer, TFBertModel

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
try:
    tpu=tf.distribute.cluster_resolver.TCPClusterResolver()# this is a TensorFlow class
    #used to create a TPUStrategy object for training on TPUs.
    
    print("Device : ",tpu.master())#returns name of TPU device designated as master (the master manages training and resources)
    tf.config.experimental_connect_to_cluster(tpu)#connect TF class to cluster
    
    tf.tpu.experimental.initialize_tpu_system(tpu)# initializing TPU system
    
    strategy=tf.distribute.experimental.TPUStrategy(tpu)# distribute learning across multiple TPUs
    
except:
    strategy=tf.distribute.get_strategy()#returns current strategy 
    # for CPU and single GPU see https://www.kaggle.com/code/anasofiauzsoy/tutorial-notebook/notebook

print("Number of replicas : ",strategy.num_replicas_in_sync)# shows copies of model undergoing training that are used to
#synchronize gradient later


print(tf.__version__)

# Number of replicas :  1
# 2.6.4

#NOTE: NUmber of replicas is 1. It means 1 copy of the model is going through the process and is going to have the gradient sync.
#NOTE: The try block most proably did not work.

Number of replicas :  1
2.17.0


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
import pandas as pd

train = pd.read_csv("data/train.csv")
train = train[:8] # for faster reproducing and fixing purposes --- make a smaller dataset

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
model_name = 'bert-base-multilingual-cased'
tokenizer = BertTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
def encode_sentence(s):
   tokens = list(tokenizer.tokenize(s))
   tokens.append('[SEP]')
   return tokenizer.convert_tokens_to_ids(tokens)

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
def bert_encode(hypotheses, premises, tokenizer):
    
  num_examples = len(hypotheses)
  
  sentence1 = tf.ragged.constant([
      encode_sentence(s)
      for s in np.array(hypotheses)])
  sentence2 = tf.ragged.constant([
      encode_sentence(s)
       for s in np.array(premises)])

  cls = [tokenizer.convert_tokens_to_ids(['[CLS]'])]*sentence1.shape[0]
  input_word_ids = tf.concat([cls, sentence1, sentence2], axis=-1)

  input_mask = tf.ones_like(input_word_ids).to_tensor()

  type_cls = tf.zeros_like(cls)
  type_s1 = tf.zeros_like(sentence1)
  type_s2 = tf.ones_like(sentence2)
  input_type_ids = tf.concat(
      [type_cls, type_s1, type_s2], axis=-1).to_tensor()

  inputs = {
      'input_word_ids': input_word_ids.to_tensor(),
      'input_mask': input_mask,
      'input_type_ids': input_type_ids}

  return inputs

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
train_input = bert_encode(train.premise.values, train.hypothesis.values, tokenizer)

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
max_len = 50
from transformers import BertTokenizer, TFBertModel


def build_model():
    bert_encoder = TFBertModel.from_pretrained(model_name)
    input_word_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_word_ids")
    input_mask = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_mask")
    input_type_ids = tf.keras.Input(shape=(max_len,), dtype=tf.int32, name="input_type_ids")
    
    embedding = bert_encoder([input_word_ids, input_mask, input_type_ids])[0]
    output = tf.keras.layers.Dense(3, activation='softmax')(embedding[:,0,:])
    
    model = tf.keras.Model(inputs=[input_word_ids, input_mask, input_type_ids], outputs=output)
    model.compile(tf.keras.optimizers.Adam(lr=1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    return model

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
with strategy.scope():
    model = build_model()
    model.summary()

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_word_ids (InputLayer  [(None, 50)]                 0         []                            
 )                                                                                                
                                                                                                  
 input_mask (InputLayer)     [(None, 50)]                 0         []                            
                                                                                                  
 input_type_ids (InputLayer  [(None, 50)]                 0         []                            
 )                                                                                                
                                                                                              

In [10]:
# --- [CELL 9]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
# === BEFORE (original) ===
# model.fit(train_input, train.label.values, epochs = 2, verbose = 1, batch_size = 64, validation_split = 0.2)

# === AFTER (edited) ===
# Ensure fixed-length BERT inputs (max_len) and avoid validation_split issues with tiny datasets
def _pad_or_truncate(x, max_len):
    x = x[:, :max_len]
    pad_len = max_len - tf.shape(x)[1]
    x = tf.pad(x, [[0, 0], [0, pad_len]])
    return x

train_input_fixed = {
    'input_word_ids': _pad_or_truncate(train_input['input_word_ids'], max_len),
    'input_mask': _pad_or_truncate(train_input['input_mask'], max_len),
    'input_type_ids': _pad_or_truncate(train_input['input_type_ids'], max_len),
}

model.fit(
    train_input_fixed,
    train.label.values,
    epochs=2,
    verbose=1,
    batch_size=4,
    validation_split=0.2
)

Epoch 1/2
2/2 [==============================] - 23s 4s/step - loss: 1.1328 - accuracy: 0.1667 - val_loss: 3.7853 - val_accuracy: 0.5000
Epoch 2/2
2/2 [==============================] - 3s 1s/step - loss: 2.7089 - accuracy: 0.5000 - val_loss: 6.7256 - val_accuracy: 0.5000


In [11]:
import numpy as np

# Recompute the true (untruncated) tokenized width implied by the raw data + tokenizer logic.
# encode_sentence already appends [SEP], and bert_encode prepends one [CLS].
true_lengths = []
for h, p in zip(np.array(train.premise.values), np.array(train.hypothesis.values)):
    l = 1 + len(encode_sentence(h)) + len(encode_sentence(p))  # 1 for [CLS]
    true_lengths.append(l)

expected_max_len = int(max(true_lengths))
actual_encoded_width = int(train_input["input_word_ids"].shape[1])
model_input_width = int(model.input_shape[0][1])

# Strong contract: model/data width must match the true tokenized max width, not an arbitrary constant.
assert actual_encoded_width == expected_max_len, (
    f"Encoded width mismatch: got {actual_encoded_width}, expected true max tokenized width {expected_max_len}"
)
assert model_input_width == expected_max_len, (
    f"Model input width mismatch: got {model_input_width}, expected {expected_max_len}"
)

AssertionError: Model input width mismatch: got 50, expected 172